# LegPy tutorial I
### Running the code

#### Fernando Arqueros, Jaime Rosado, Victor Moya


LegPy (Low energy gamma-ray simulation in Python) is a Monte Carlo (MC) simulation code for the transportation of gamma rays, electrons and positrons with energies up to few MeVs (depending of the medium).

Several geometries are supported (cylinder, orthohedron and sphere). Photons, electrons or positrons are produced as paralell (or divergent) beams or from isotropic sources with energies following several spectral distributions (monoenergetic, flat, exponential, etc.) including those of realistic X-ray and gamma ray sources. Particles can be transported either through a single material or in geometries filled with two different media.

The description of the MC algorithm and the validation of the various approximations are described in https://doi.org/10.1016/j.radmeas.2023.107029 and https://doi.org/10.48550/arXiv.2501.15979. The latest implementations including e-/e+ pair creation and positrons transportation will be described in a foreseen publication.

This tutorial explains the procedure to run a simulation with the LegPy package. It will be shown that only five code lines in this notebook are sufficient. Other tutorials (**LegPy tutorial II** and **III**) present a number of examples to visualize and analyze the results of the simulation for **photon** and **electron/positron** beams, respectively. 

## Previous steps

If you have downloaded LegPy in you PC and are runnig this notebook with Jupyter or a similar application, ignore this step. But if you are running this notebook from Google Colab, first you should clone the last version of the LegPy repository each time you inizialite Colab and add the directory where LegPy is located to the Python search path:

In [ ]:
#Only for Colab users!!
!git clone https://github.com/JaimeRosado/LegPy.git
import sys
sys.path.insert(0,'/content/LegPy')

Then, import the necessary modules to run this notebook:

In [ ]:
import LegPy as lpy
import numpy as np
import matplotlib.pyplot as plt

The user has to "construct" four objects that are the main ingredients of the MC simulation:

- The composition of the medium and their physical properties at microscopic level.
- The geometry of the medium.
- The energy spectrum of the beam crossing the medium.
- The geometry of the beam and type of particles (photons, electrons or positrons)

Let's start.

## Construction of the medium I: Composition

LegPy provides the necessary data related to the interaction of photons and electrons/positrons with a large number of atoms, compounds and mixtures from the National Institute of Standards and Technology (NIST Database) https://www.nist.gov/.

Select your media from this list

In [ ]:
lpy.List_Media('Ic')

You also may search a medium name containing a string

In [ ]:
lpy.List_Media('water')

**A) Construct the medium providing the following data:**

- `name` (as written in the above list): 'Al', 'Ge', 'Bone, Cortical (Icrp)', 'Sodium Iodide', 'C-552 Air-Equivalent Plastic', 'Water, Liquid'...
- `density` (g/cm$^3$): Optional. Default value is stored in data files. Different values can be used for testing purposes.

In [ ]:
medium = lpy.Medium(name='Aluminum Oxide')
#medium = lpy.Medium(name='Al', density=2.7)
#medium = lpy.Medium(name='Bone, Cortical (Icrp)', density=1.8)

In case of a simulation on a **two-media** geometry, both have to be constructed:

In [ ]:
medium_1 = lpy.Medium(name='Aluminum Oxide')
medium_2 = lpy.Medium(name='Al', density=2.7)

### Plot attenuation coefficients of photons vs E

You can plot the components of the attenuation coeficients vs energy of a medium. In the next command you have to provide:

- `energies` : array of energies in MeV, better in log scale.
- `l_style` : line style for plotting ('', ':', etc.), defalut '' (solid line).
- `ph` : True or False, default to True.
- `inc` : True or False, default to True.
- `coh` : True or False, default to True.
- `pair` : True or False, default to True.
- `en-abs` : True or False, default to True.
- `tot` : True or False, default to True.

Several media can be plotted in the same figure so you can define several media above (with different names) and compare attenuation coefficients.

This step is optional so you can skip it.

In [ ]:
E1 = 0.01 # MeV
E2 = 20. # MeV
energy_range = np.logspace(np.log10(E1), np.log10(E2), num=150) # 150 points in a log-scale E(MeV) between E1 and E2
medium.plot_mu(energies=energy_range)
#medium2.plot_mu(energies=energy_range, l_style=':', tot=True)

### Plot CSDA ranges of electrons and positrons vs E

You can plot the continuous slowing down approximation (CSDA) range vs energy of the selected media. In the next command you have to provide:

- `energies` : array of energies in MeV, better in log scale.
- `l_style` : line style for plotting ('', ':', etc.), defalut '' (solid line).
- `units` : 'cm' or 'gcm2'. Default is 'cm'.

Several media can be plotted in the same figure so you can define several media above (with different names) and compare ranges (in the same units).

This step is optional so you can skip it.

In [ ]:
E1 = 0.01 # MeV
E2 = 20. # MeV
energy_range = np.logspace(np.log10(E1), np.log10(E2), num=150) # 150 points in a log-scale E(MeV) between E1 and E2
medium.plot_R(energies=energy_range)
#medium2.plot_R(energies=energy_range, l_style = ':')

## Construction of the medium II: Geometry

Several geometries are available: cylinder, orthohedron and sphere.

**Note**: *This geometry can be filled either with **one** single **medium** or contained **two media** splitted by an interface surface. The location of this surface has to be given for the construction of the geometry object*

**B) Select your geometry**

### Cylinder





Cylinder oriented with its axis along the z axis and its base at z=0. You have to provide:

- `r` or `diam` (cm): radius or diameter.
- `z` (cm): height.

In the **two-media** case the interface can be a horizontal surface at a given depth:

- `z_int` (cm): depth of the interface, that is, medium_1 in [0, z_int] and medium_2 in [z_int, z].

or a cylinder with a given radius:

- `r_int` (cm): medium_1 in the radial interval [0, r_int] and medium_2 in [r_int, r].

For this geometry, you may choose either cylindrical or cartesian voxelization for the energy deposit matrix. Cylindrical voxelization is appropriate for parallel beams along the z axis and isotropic sources located at the z axis. In this case, you have to input the number of intervals along the coordinates r and z:

- `n_r`.
- `n_z`.

Cartesian voxelization can also be applied in any situation and medium geometry. Here, you have to provide:

- `n_x`.
- `n_y`.
- `n_z`.

Choose your option and construct the geometry.

In [ ]:
geometry = lpy.Geometry(name='cylinder', z=2., r=1., n_z=50, n_r=1) # Cylindrical voxelization
#geometry = lpy.Geometry(name='cylinder', z=3., r=2., r_int = 0.5, n_x=10, n_y=10, n_z=10) # Cartesian voxelization

### Orthohedron

Rectangular parallelepiped oriented with its longitudinal axes parallel to the x, y, z axes. The center of bottom side is assumed to be at the origin of coordinates (0,0,0). In this geometry, only the cartesian voxelization is supported. You have to provide the dimensions of the orthohedron and the number of intervals along each axis:

- `x` (cm).
- `y` (cm).
- `z` (cm).

In the **two-media** case the interface is a horizontal surface at a given depth:

- `z_int` (cm): depth of the interface, that is, medium_1 in [0, z_int] and medium_2 in [z_int, z].

Cartesian voxelization is used in this geometry. Here, you have to provide:
- `n_x`.
- `n_y`.
- `n_z`.

In [ ]:
geometry = lpy.Geometry(name='orthohedron', x=10., y=10., z=10., z_int = 3., n_x=10, n_y=10, n_z=10)

### Sphere





Sphere centered at the origin of coordinates (0,0,0). Both cartesian and spherical voxelization can be chosen. So you have to provide either:

- `r` or `diam` (cm).
- `n_r`.

or:

- `r` (cm).
- `n_x`.
- `n_y`.
- `n_z`.

In the **two-media** case the interface can be a flat surface at a given depth:

- `z_int` (cm): z location of the interface, that is, medium_1 in the z interval [-r, z_int] and medium_2 in [z_int, r].

or a spherical surface with a given radius:

- `r_int` (cm): medium_1 in the sphere [0, r_int] and medium_2 in [r_int, r].

In [ ]:
geometry = lpy.Geometry(name='sphere', r=15.72, n_r=15) # Spherical voxelization
#geometry = lpy.Geometry(name='sphere', diam=10., r_int = 3., n_x=10, n_y=10, n_z=10) # Cartesian voxelization

#### Plot the geometry


Plot the geometry in the reference coordinate system. This step is optional.

In [ ]:
geometry.plot();

## Construction of the beam I:  Energy spectrum

**C) Select the energy spectrum of the particle beam**

from one of the following options:



### Monoenergetic



Input parameters:
- `E` (MeV).

In [ ]:
spectrum = lpy.Spectrum(name = 'mono', E = 1.)

### Multi-monoenergetic




Input parameters:
- `E_w`: energies (MeV) and their corresponding weights in a numpy array.

In [ ]:
E_w = np.array([[0.511, .80], [1.25, 0.20]]) # [[E1, w1], [E2, w2],....]
spectrum = lpy.Spectrum(name = 'multi_mono', E_w = E_w)



### Flat





Input parameters:
- `E_min` (MeV).
- `E_max` (MeV).

In [ ]:
spectrum = lpy.Spectrum(name = 'flat', E_min = 0.1, E_max = 1.0)



### Gaussian profile.





Input parameters:
- `E_mean` (MeV).
- `E_sigma` (MeV).

Internal cut: 2 x E_mean > E > 0.

In [ ]:
spectrum = lpy.Spectrum(name = 'gaussian', E_mean = 0.5, E_sigma = 0.03)



### Exponential





$I(E) \propto  e^{-E/E_{ch}}$, with E_min < E < E_max.

Input parameters:

- `E_min` (MeV).
- `E_max` (MeV).
- `E_ch` (MeV).

In [ ]:
spectrum = lpy.Spectrum(name = 'exponential', E_min = 0.1, E_max = 1.0, E_ch = 0.5)



### Reciprocal




$ I(E) \propto \frac{1}{E} $, with E_min < E < E_max.

Input parameters:
- `E_min` (MeV).
- `E_max` (MeV).

In [ ]:
spectrum = lpy.Spectrum(name = 'reciprocal', E_min = 0.01, E_max = 15.)



### From a file






A number of realistic spectra are provided in LegPy

- X-Ray spectra generated with SpekPy Web 2.08: For example 80kV_3.0Al is for a tube potential of 80kV on Tungsten (default) with a filter of 3.0 mm Aluminum. In 60kV-Mo_0.04Mo both target and filter are Molybdenum.
- Gamma-ray spectra of clinical accelerators: For example 10MV_VARIAN is a photon spectrum of a VARIAN accelerator of 10MV, 6MeV_VARIAN is an electron spectrum of 6MeV. More details of these spectra are given in the header of the corresponding .txt file inside the folder LegPy/beam_spectra.

Check the available spectra:

In [ ]:
lpy.List_Spectra()

and select one of them:

In [ ]:
spectrum = lpy.Spectrum(name = 'from_file', file = '6MV_Elekta')

If you want to use your own spectrum file, you should pass its path to the parameter `file`. The file must be a txt file with two columns: Energy (MeV)   ------   Relative Intensity (au)

In [ ]:
#This only works if there is a spectrum file called my_spectrum.txt in the working directory
#spectrum = lpy.Spectrum(name = 'from_file', file = 'my_spectrum')

#### Plot the spectrum







You can plot the energy spectrum of incident beam. Again, just to check it is OK.

A number of photons are generated randomly following the requested spectrum in logaritmic scale. You should input:
- `n_part` : number of particles, default to 10^5.
- `n_bin` : number of intervals, default to 50.

In [ ]:
spectrum.plot(n_part = 100000, n_bin = 50)

## Construction of the beam II:  Geometry

**D) Select the particle (photon or electron) and the geometrical properties of the beam.**

Input parameter in all beam geometries:
- `particle`= 'photon', 'electron' or 'positron'. Default is `particle` = 'photon'.

NOTE: In order not to waste computing time the beam geometry has to be defined in such a way that all particles reach the medium.

### Parallel beam

Parallel beam with entrance plane perpendicular to z axis. In general not applicable for the sphere.

Input parameters:

- `theta` (degrees) : zenith angle from z axis, default to 0.
- `phi` (degrees) : azimuth angle from x axis, default to 0.
- `p_in` (cm) : numpy array with the coordinates of the center of the beam cross section at the entrance plan, default to (0,0,0).
- `diam` (cm) : beam diameter, default to 0. (i.e., pencil beam).

In [ ]:
beam = lpy.Beam(name = 'parallel')
#beam = lpy.Beam(particle='positron', name = 'parallel')
#beam = lpy.Beam(name = 'parallel', theta = 15.0, phi = 30.0, p_in = np.array([0.1, -0.1, 0.0]))

###  Divergent beam / Isotropic source

Three options are available:

1) **Divergent beam** with the focus located on the z(<0) axis. Not applicable for the sphere.

Input parameters:

- `p_in` (cm): numpy array with the coordinates of the focus location below the XY plane (z<0). Default to x=y=0 (source on the z axis). Small (x,y) values are also possible but might increase the computing time.      
- size of the field located on the XY plane, centered at (0,0):
    - if circular, provide `diam` (cm)
    - if rectangular, provide `x_ap`, `y_ap` (cm) size

In [ ]:
z = 25. # cm
diam = 1. # cm
beam = lpy.Beam(name = 'isotropic', diam = diam, p_in = np.array ([0., 0., -z]))
#beam = lpy.Beam(name = 'isotropic',  particle='electron', x_ap = 1., y_ap = 0.5, p_in = np.array ([1., 0., -z]))

2) Isotropic **source** located **outside the medium**. Not applicable for the sphere. Currently only available for point sources. 

It is geometrically equals to the **divergent beam**

Input parameters:
- `p_in` (cm): numpy array with the coordinates of the source location below the XY plane (z<0). Default to x=y=0 (source on the z axis). Small (x,y) values are also possible but might increase the computing time.
- size of the entrance aperture located on the XY plane, centered at (0,0):
    - if circular, provide `diam` (cm)
    - if rectangular, provide `x_ap`, `y_ap` (cm) size

In [ ]:
z = .5 # cm
x, y = 0.05, -0.01 # cm
beam = lpy.Beam(name = 'isotropic', diam = 1., p_in = np.array ([x, y, -z]))
#beam = lpy.Beam(name = 'isotropic', particle='electron', x_ap = 1., y_ap = 0.5, p_in = np.array ([0., 0., -z]))

3) Isotropic **source** located **inside the medium**.

Both point sources and sources uniformly distributed in a sphere, cylinder or orthohedron are allowed. 

Input parameters:

- `p_in` (cm): numpy array with the coordinates of the location of a point source (or the center of a finite size  source). Default to p_in = [0., 0., 0.] (source on the coordinates origin).
- size of the source. Default is point source
    - if orthohedron, provide `x_s`, `y_s`, `z_s` (cm) size. Only one of these parameters are allowed to be 0.
    - if cylinder, provide `r_s`, `z_s` (cm) size. Null radius `r_s` is allowed.  
    - if sphere, provide `r_s`, (cm).

In [ ]:
x, y, z = 0.3, -0.3, 0.5 # cm
beam = lpy.Beam(name = 'isotropic', particle='photon', p_in = np.array ([x, y, z]), r_s = 5.)
#beam = lpy.Beam(name = 'isotropic')
#beam = lpy.Beam(name = 'isotropic', particle='electron', p_in = np.array ([0., 0., 0.]), z_s = 1., r_s = 5.)

#### Plot a few tracks


Check a few (50) beam tracks into the medium with the geometry you have just constructed. Optional

In [ ]:
lpy.Plot_beam(medium, geometry, spectrum, beam)

## Monte Carlo Simulation


It transports the particle beam through the medium (or media) defined above.

Parameters to be provided:

For all particle beams, **photons, electrons and positrons**:  

- `n_part`: Number of particles (simulation cases). Default is `n_part` = 100.
- `E_cut`: Energy cut in MeV. Default is `E_cut` = 0.01 MeV.
- `tracks`: Plot tracks of the particle beam (not advised for > 100 particles). Default is `tracks` = False.
- `n_ang`: Number of intervals to construct angular histograms of particles leaving the medium. Default is 20.
- `n_E`: Number of energy intervals to construct several histograms. It applies to histogram of photons leaving the medium, spectra of absorbed energy and spectral fluence. Default is 20.
- `x_ray`: If False, X-rays from atomic de-excitation after photoelectric absorption are neglected and the whole photon energy is absorbed in the interaction point. Default is `x_ray` = True. Only K$\alpha$ and K$\beta$ photons are generated.

In case of a **photon** beam:

- `n_zloc` : Number of locations along the z axis to evaluate the photon fluence if `fluence` = z (see next parameter). Default is 20. Note that `n_zloc` is different from `n_z` defined in the `Geometry` object
- `fluence`: If fluence not False (default), it can take two values. If `fluence` = z, the photon fluence is only calculated along the main axis in n_zloc+1 locations (from z = 0 to z = maximum depth) on surfaces of the size of the cross section of the voxels defined in `Geometry`. If `fluence` = True, the photon fluence is calculated in the whole medium volume along surfaces perpendicular to the z axis in locations given by the voxelization defined in `Geometry`.
- `points` : Plot interaction points (not advised for > 100 photons). Default is `points` = False.
- `e_transport`: If True, all secondary electrons (photoelectric absorption and Compton scattering) are transported. By default (`e_transport` = False) electron energy is absorbed at the interaction point.

In case of an **electron** or **positron** beam:

- `n_zloc`: Number of z positions/depths to construct transmission plots. Default is 20.
- `e_length`, `e_K` : The electron/positron transport is performed by steps of either same length (`e_length`) or same energy loss fraction (`e_K`). By default, e_length is obtained from the voxel size, but these parameters may also be specified, if desired. The parameter `e_length` is in $\mu$m.

Next, some examples:

Simulation on **one** single **medium**:

In [ ]:
result = lpy.MC(medium, geometry, spectrum, beam, n_part=1000, n_E=50)
#result = lpy.MC(medium, geometry, spectrum, beam, n_part=10000, e_transport = True)

In case of a **two-media** geometry:

In [ ]:
result = lpy.MC([medium_1, medium_2], geometry, spectrum, beam, n_part=1000)
#result = lpy.MC([medium_1, medium_2], geometry, spectrum, beam, n_part=int(1.e5), n_zloc = 100)

Return:

An object (dubbed **result**) containing

If **photon** beam:

- the spatial distribution of deposited energy
- the angular and energy spectra of outgoing photons
- the histogram of absorbed energy
- the spectral fluence along the z axis.

If **electron/positron** beam:

- the spatial distribution of deposited energy
- the histogram of electron/positron ranges
- the angular histogram of backscattered electrons/positrons.

Once this object is generated, this information can be plotted or stored in files (see examples below).


In **LegPy tutorial II and III** a number of examples can be found showing the procedure to plot an analyze the results of the simulation of photon and electron/positron beams respectively. 